# Sleep Deprivation: pySCENIC GRN + CellOracle In Silico Perturbation
## Run in Google Colab with GPU runtime

**Before running:**
1. Upload these files to your Google Drive (in a folder called `sleep_deprivation`):
   - `GSE137665_processed.h5ad` (62 MB)
   - `GSE137665_raw.h5ad` (37 MB)
2. Runtime → Change runtime type → T4 GPU
3. Run cells in order

In [ ]:
# ============ Cell 1: Install dependencies ============
!pip install -q scanpy pandas numpy matplotlib seaborn scipy anndata
!pip install -q pyscenic celloracle arboreto
!pip install -q numba scikit-learn networkx
!pip install -q decoupler  # TF activity inference (backup)

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

sc.settings.set_figure_params(dpi=100, facecolor='white')
print('All packages installed!')

In [ ]:
# ============ Cell 2: Mount Google Drive & Load Data ============
from google.colab import drive
drive.mount('/content/drive')

# Path to your uploaded files
DATA_DIR = '/content/drive/MyDrive/sleep_deprivation'

# Load processed data
adata = sc.read_h5ad(f'{DATA_DIR}/GSE137665_processed.h5ad')
print(f'Processed: {adata.n_obs} cells x {adata.n_vars} genes')
print(f'Cell types: {adata.obs["cell_type"].value_counts().to_dict()}')

# Load raw data for full expression
raw = sc.read_h5ad(f'{DATA_DIR}/GSE137665_raw.h5ad')
raw = raw[raw.obs_names.isin(adata.obs_names)]
print(f'Raw: {raw.n_obs} cells x {raw.n_vars} genes')

In [ ]:
# ============ Cell 3: Focus on Hypothalamus + Prepare for SCENIC ============
# pySCENIC needs raw-ish counts (normalized but not scaled)
hypo = raw[raw.obs['brain_region'] == 'Hypothalamus'].copy()
sc.pp.normalize_total(hypo, target_sum=1e4)
sc.pp.log1p(hypo)
print(f'Hypothalamus: {hypo.n_obs} cells, {hypo.n_vars} genes')

# For SCENIC: filter to cells with Pomc expression or nearby clusters
# This reduces compute while keeping biological signal
# Take top 3000 variable genes + all known TFs
sc.pp.highly_variable_genes(hypo, n_top_genes=3000)

# SCENIC works better with raw counts (not log-normalized)
# Use the .raw attribute if available, or re-normalize without log
hypo_raw = raw[raw.obs['brain_region'] == 'Hypothalamus'].copy()
sc.pp.normalize_total(hypo_raw, target_sum=1e4)
# Keep as linear (no log) for SCENIC

print(f'Ready for SCENIC: {hypo_raw.n_vars} genes total')

In [ ]:
# ============ Cell 4: Download SCENIC reference databases ============
!mkdir -p scenic_data

# Mouse TF list (from AnimalTFDB / SCENIC)
!wget -q -O scenic_data/mm_mgi_tfs.txt \
  https://raw.githubusercontent.com/aertslab/pySCENIC/master/resources/mm_mgi_tfs.txt \
  || echo 'Downloading from mirror...'

# Alternative: Download TF list from SCENIC resources
import urllib.request
tf_url = 'https://resources.aertslab.org/cistarget/tf_lists/allTFs_mm.txt'
try:
    urllib.request.urlretrieve(tf_url, 'scenic_data/mm_mgi_tfs.txt')
    print('TF list downloaded')
except:
    print('Using built-in TF list')
    # Fallback: use known mouse TFs
    MOUSE_TFS = ['Nr3c1','Crem','Fos','Fosb','Jun','Junb','Jund','Atf3','Atf4',
        'Mef2c','Mef2d','Srebf1','Srebf2','Nr4a1','Nr4a2','Nr4a3',
        'Egr1','Egr2','Egr3','Klf4','Klf9','Sox2','Sox9','Mafb','Cebpb',
        'Hlf','Zic1','Npas4','Per1','Per2','Cry1','Cry2','Clock','Arntl',
        'Nfil3','Dbp','Tef','Stat3','Stat5a','Nfkb1','Rela','Sp1','Sp3',
        'Yy1','Ctcf','Pparg','Ppargc1a','Foxo1','Foxo3','Rest','Mecp2',
        'Olig1','Olig2','Sox10','Tbr1','Neurod1','Neurod2','Neurod6',
        'Lhx6','Dlx1','Dlx2','Nkx2-1','Sim1','Bcl11b','Satb2',
        'Rxrg','Rorb','Rora','Rorc','Nr1d1','Nr1d2','Thrb','Ppara',
        'Nr5a1','Nr5a2','Esr1','Esr2','Ar']
    with open('scenic_data/mm_mgi_tfs.txt', 'w') as f:
        for tf in MOUSE_TFS:
            f.write(tf + '\n')

# Motif ranking database (~1GB for mouse)
# This is the largest file - may take 5-10 min to download
print('Downloading motif ranking database (~1GB)...')
motif_url = 'https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc9nr/gene_based/mm10__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather'
!wget -q --show-progress -O scenic_data/mm10_rankings.feather '{motif_url}' 2>&1 || echo 'Download failed - will use simplified approach'

# Motif-to-TF annotation
anno_url = 'https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl'
!wget -q -O scenic_data/motifs.tbl '{anno_url}' 2>&1 || echo 'Annotation download failed'

print('Database download complete (check for errors above)')

In [ ]:
# ============ Cell 5: Export expression matrix for pySCENIC ============
# pySCENIC CLI expects a CSV: rows=genes, cols=cells

# Subset to highly variable genes + TFs for manageable size
from scipy.sparse import issparse

# Get expression matrix
if issparse(hypo.X):
    expr_df = pd.DataFrame(hypo.X.toarray(), index=hypo.obs_names, columns=hypo.var_names)
else:
    expr_df = pd.DataFrame(hypo.X, index=hypo.obs_names, columns=hypo.var_names)

# Transpose: genes x cells (SCENIC format)
expr_scenic = expr_df.T
print(f'Expression matrix for SCENIC: {expr_scenic.shape}')

# Save as parquet (faster) and CSV
expr_scenic.to_parquet('expr_matrix.parquet')
expr_scenic.to_csv('expr_matrix.csv')
print('Expression matrix saved (CSV + parquet)')

In [ ]:
# ============ Cell 6: Run pySCENIC GRN inference ============
import os.path as op

# Check if databases exist
rankings_path = 'scenic_data/mm10_rankings.feather'
tf_list_path = 'scenic_data/mm_mgi_tfs.txt'
motif_anno_path = 'scenic_data/motifs.tbl'

if op.exists(rankings_path) and op.exists(tf_list_path):
    print('Running full pySCENIC pipeline...')
    
    # Step 1: GRNBoost2 - co-expression network
    !pyscenic grn expr_matrix.parquet \
        {tf_list_path} \
        -o scenic_adjacencies.csv \
        --num_workers 4 \
        --seed 42
    
    print('GRN inference complete!')
    
    if op.exists(motif_anno_path):
        # Step 2: RcisTarget - prune with motif evidence
        !pyscenic ctx scenic_adjacencies.csv \
            {rankings_path} \
            --annotations_fname {motif_anno_path} \
            --expression_mtx_fname expr_matrix.parquet \
            --output scenic_regulons.csv \
            --num_workers 4
        
        print('RcisTarget regulon pruning complete!')
        
        # Step 3: AUCell - TF activity per cell
        !pyscenic aucell expr_matrix.parquet \
            scenic_regulons.csv \
            --output scenic_aucell.csv \
            --num_workers 4
        
        print('AUCell scoring complete!')
else:
    print('SCENIC databases not available.')
    print('Running simplified GRN analysis instead...')
    
    # Simplified approach: use arboreto GRNBoost2 without motif pruning
    from arboreto.algo import grnboost2
    
    # Load expression
    expr = pd.read_parquet('expr_matrix.parquet')
    
    # Load TF list
    with open(tf_list_path) as f:
        tfs = [line.strip() for line in f if line.strip()]
    tfs_present = [t for t in tfs if t in expr.index]
    print(f'TFs available: {len(tfs_present)}/{len(tfs)}')
    
    # Run GRNBoost2 (co-expression + tree-based regression)
    # This is the core of pySCENIC step 1
    links = grnboost2(expression_data=expr, tf_names=tfs_present, verbose=True)
    links.columns = ['TF', 'target', 'importance']
    links.to_csv('scenic_adjacencies.csv', index=False)
    print(f'GRNBoost2 found {len(links)} regulatory links')
    print(links.head(10))

In [ ]:
# ============ Cell 7: Analyze SCENIC results - focus on Pomc ============

# Load the GRN links
links = pd.read_csv('scenic_adjacencies.csv')
print(f'Total regulatory links: {len(links)}')

# Filter for Pomc-related links
pomc_targets = links[links['TF'] == 'Pomc'] if 'Pomc' in links['TF'].values else pd.DataFrame()
pomc_regulators = links[links['target'] == 'Pomc'] if 'Pomc' in links['target'].values else pd.DataFrame()

print(f'Pomc -> target genes: {len(pomc_targets)}')
print(f'Regulators -> Pomc: {len(pomc_regulators)}')

if len(pomc_regulators) > 0:
    print('\nTop predicted regulators of Pomc:')
    for _, row in pomc_regulators.sort_values('importance', ascending=False).head(10).iterrows():
        print(f'  {row["TF"]:15s}  importance={row["importance"]:.3f}')

if len(pomc_targets) > 0:
    print('\nTop predicted targets of Pomc:')
    for _, row in pomc_targets.sort_values('importance', ascending=False).head(15).iterrows():
        print(f'  {row["target"]:15s}  importance={row["importance"]:.3f}')

# Top regulators in the dataset (most connected TFs)
tf_connectivity = links.groupby('TF').agg(
    n_targets=('target', 'nunique'),
    mean_importance=('importance', 'mean')
).sort_values('n_targets', ascending=False)

print(f'\nTop 10 most connected TFs:')
print(tf_connectivity.head(10))

In [ ]:
# ============ Cell 8: In Silico Pomc Perturbation (Local GRN Model) ============
# Using CellOracle approach: propagate perturbation through GRN

import networkx as nx
from scipy.stats import spearmanr

# Build a local GRN around Pomc using SCENIC links + Spearman correlations
print('Building Pomc-centered perturbation model...')

# Get top N TFs and targets connected to Pomc
pomc_connected = set()
if len(pomc_regulators) > 0:
    pomc_connected.update(pomc_regulators.nlargest(10, 'importance')['TF'].tolist())
if len(pomc_targets) > 0:
    pomc_connected.update(pomc_targets.nlargest(20, 'importance')['target'].tolist())
pomc_connected.add('Pomc')

# Also add top TFs from the global network
top_global_tfs = tf_connectivity.head(10).index.tolist()
pomc_connected.update(top_global_tfs)

pomc_connected = list(pomc_connected)
print(f'Perturbation network: {len(pomc_connected)} genes')

# Build adjacency matrix from expression correlations
expr = pd.read_parquet('expr_matrix.parquet')
core_genes = [g for g in pomc_connected if g in expr.index]
print(f'Genes in expression matrix: {len(core_genes)}')

core_expr = expr.loc[core_genes].values
n_genes = len(core_genes)

# Spearman correlation matrix
adj = np.zeros((n_genes, n_genes))
for i in range(n_genes):
    for j in range(i+1, n_genes):
        if np.std(core_expr[i]) > 0 and np.std(core_expr[j]) > 0:
            r, _ = spearmanr(core_expr[i], core_expr[j])
            adj[i, j] = r
            adj[j, i] = r

# Get Pomc position
pomc_pos = core_genes.index('Pomc')

# Perturbation simulation
# Method 1: Direct correlation-based (simple)
pomc_mean = core_expr[pomc_pos].mean()
direct_effects = []
for j, gene in enumerate(core_genes):
    if gene == 'Pomc':
        continue
    r = adj[pomc_pos, j]
    pred_fc = -r * pomc_mean  # Removing Pomc
    direct_effects.append({
        'gene': gene,
        'correlation': r,
        'predicted_log2FC': pred_fc,
        'direction': 'Down' if pred_fc < 0 else 'Up'
    })

perturb_df = pd.DataFrame(direct_effects).sort_values('predicted_log2FC', key=abs, ascending=False)

# Method 2: Network propagation (simulate signal cascade)
# This is more like CellOracle's approach
from scipy.linalg import expm

# Create normalized adjacency
np.fill_diagonal(adj, 0)
D_inv = np.diag(1.0 / (np.sum(np.abs(adj), axis=1) + 1e-8))
L = D_inv @ adj  # Random walk Laplacian

# Initial perturbation: set Pomc to -1 (remove)
delta = np.zeros(n_genes)
delta[pomc_pos] = -pomc_mean

# Propagate through network (2 steps)
propagated = delta.copy()
for step in range(2):
    propagated = 0.7 * L @ propagated + 0.3 * delta

print('\nNetwork propagation results:')
for j, gene in enumerate(core_genes):
    if gene != 'Pomc' and abs(propagated[j]) > 0.01:
        print(f'  {gene:15s}  propagated_effect={propagated[j]:+.3f}')

In [ ]:
# ============ Cell 9: CellOracle-style Vector Field Perturbation ============
# If CellOracle is installed and working

try:
    import celloracle as co
    from celloracle.applications import Oracle_development_module as odm
    print(f'CellOracle version: {co.__version__}')
    
    # Initialize Oracle
    oracle = co.Oracle()
    
    # Load our data
    oracle.import_anndata_as_raw_count(
        adata=hypo,
        cluster_column_name='leiden',
        embedding_name='X_umap'
    )
    print('Oracle initialized with AnnData')
    
    # Use our SCENIC links as GRN
    scenic_links = pd.read_csv('scenic_adjacencies.csv')
    if 'TF' in scenic_links.columns:
        scenic_links.columns = ['source', 'target', 'importance']
    oracle.import_TF_data(TF_info_matrix=scenic_links)
    
    # Fit GRN model
    oracle.perform_PCA()
    oracle.fit_GRN_for_simulation(alpha=10, use_linear=True)
    print('GRN model fitted')
    
    # In silico Pomc knockout
    oracle.simulate_shift(
        perturb_condition={'Pomc': 0.0},  # Knock out Pomc
        n_propagation=3
    )
    print('Pomc knockout simulated!')
    
    # Get predicted changes
    oracle.estimate_transition_prob(n_neighbors=200, knn_random=True)
    oracle.calculate_embedding_shift()
    
    # Get differential expression prediction
    pred_genes = oracle.get_perturbation_result()
    print(f'Predicted gene expression changes: {len(pred_genes)} genes')
    print(pred_genes.head(15))
    
    # Save results
    pred_genes.to_csv('celloracle_pomc_ko_prediction.csv')
    
except ImportError as e:
    print(f'CellOracle not available: {e}')
    print('Using our network propagation results instead (Cell 8)')
except Exception as e:
    print(f'CellOracle error: {e}')
    print('Using our network propagation results instead (Cell 8)')

In [ ]:
# ============ Cell 10: Visualize Results ============

# 10.1: SCENIC GRN around Pomc
import networkx as nx

fig, ax = plt.subplots(figsize=(14, 12))

# Build network from top Pomc regulators + targets
G = nx.Graph()
G.add_node('Pomc', type='target_gene')

# Add edges from SCENIC links
pomc_edges = []
if len(pomc_regulators) > 0:
    for _, row in pomc_regulators.head(8).iterrows():
        G.add_node(row['TF'], type='tf')
        G.add_edge(row['TF'], 'Pomc', weight=row['importance'], sign='regulates')
        pomc_edges.append((row['TF'], 'Pomc'))

if len(pomc_targets) > 0:
    for _, row in pomc_targets.head(12).iterrows():
        G.add_node(row['target'], type='target')
        G.add_edge('Pomc', row['target'], weight=row['importance'], sign='targets')
        pomc_edges.append(('Pomc', row['target']))

# Layout
pos = nx.spring_layout(G, k=2, iterations=100, seed=42)

# Node colors
node_colors = []
node_sizes = []
for node in G.nodes():
    if node == 'Pomc':
        node_colors.append('#E64B35')
        node_sizes.append(2000)
    elif G.nodes[node].get('type') == 'tf':
        node_colors.append('#4DBBD5')
        node_sizes.append(1200)
    else:
        node_colors.append('#CCCCCC')
        node_sizes.append(800)

edge_colors = ['#E64B35' if e[1] == 'Pomc' else '#4DBBD5' for e in G.edges()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=2, alpha=0.5, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=ax)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#E64B35', markersize=15, label='Pomc'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#4DBBD5', markersize=12, label='Regulatory TF'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#CCCCCC', markersize=10, label='Target Gene'),
], loc='upper left', frameon=False)
ax.set_title('Pomc Regulatory Network (pySCENIC GRN)', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('SCENIC_Pomc_GRN.png', dpi=300, bbox_inches='tight')
plt.show()
print('GRN figure saved!')

In [ ]:
# 10.2: Perturbation Prediction
fig, ax = plt.subplots(figsize=(10, 6))

top_perturb = perturb_df.head(20)
colors = ['#E64B35' if x < 0 else '#4DBBD5' for x in top_perturb['predicted_log2FC']]

ax.barh(range(len(top_perturb)), top_perturb['predicted_log2FC'].values[::-1], color=colors[::-1])
ax.set_yticks(range(len(top_perturb)))
ax.set_yticklabels(top_perturb['gene'].values[::-1], fontsize=9)
ax.set_xlabel('Predicted log2 Fold Change (Pomc KO)')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('In Silico Pomc Knockout: Predicted Downstream Effects', fontsize=13, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.savefig('SCENIC_Pomc_KO_Prediction.png', dpi=300, bbox_inches='tight')
plt.show()
print('KO prediction figure saved!')

In [ ]:
# 10.3: TF Activity Heatmap (if AUCell was run)
aucell_path = 'scenic_aucell.csv'
if os.path.exists(aucell_path):
    aucell = pd.read_csv(aucell_path, index_col=0)
    print(f'AUCell matrix: {aucell.shape}')
    
    # Top variable TFs across conditions
    aucell_var = aucell.var(axis=1).sort_values(ascending=False)
    top_tfs = aucell_var.head(15).index.tolist()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Add condition annotations
    cell_order = aucell.columns
    conditions = [hypo.obs.loc[c, 'condition_code'] if c in hypo.obs_names else '?' for c in cell_order]
    
    sns.heatmap(aucell.loc[top_tfs], cmap='RdBu_r', center=0,
                xticklabels=False, ax=ax, cbar_kws={'label': 'TF Activity (AUC)'})
    ax.set_title('Top TF Activities Across Cells (pySCENIC AUCell)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('SCENIC_TF_Activity_Heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('AUCell results not available (motif pruning was skipped)')

In [ ]:
# ============ Cell 11: Compare with our local GRN results ============
# Load local results for comparison (if available)

print('=== SCENIC vs Local GRN Comparison ===')

# Our local key genes
local_key_genes = ['Pomc', 'Fos', 'Jun', 'Egr1', 'Nr4a1', 'Nr3c1', 'Crem',
                    'Clock', 'Per1', 'Per2', 'Dbp', 'Rbm3', 'Tsc22d3',
                    'Nr1d1', 'Bdnf', 'Npas4', 'Homer1', 'Arc', 'Srebf1']

# Check if they're in SCENIC results
for gene in local_key_genes:
    regulators = links[links['target'] == gene] if gene in links['target'].values else pd.DataFrame()
    targets = links[links['TF'] == gene] if gene in links['TF'].values else pd.DataFrame()
    print(f'{gene:12s}: {len(regulators)} regulators, {len(targets)} targets')

# Save all results for download
results = {
    'scenic_links': links,
    'pomc_regulators': pomc_regulators,
    'pomc_targets': pomc_targets,
    'perturbation_prediction': perturb_df,
    'tf_connectivity': tf_connectivity
}

for name, df in results.items():
    if isinstance(df, pd.DataFrame) and len(df) > 0:
        df.to_csv(f'{name}.csv', index=False if name != 'tf_connectivity' else True)
        print(f'Saved: {name}.csv')

print('\nAll results saved! Download from Files panel.')

In [ ]:
# ============ Cell 12: Download all results as ZIP ============
!zip -r scenic_results.zip \
    SCENIC_Pomc_GRN.png \
    SCENIC_Pomc_KO_Prediction.png \
    SCENIC_TF_Activity_Heatmap.png \
    scenic_links.csv \
    pomc_regulators.csv \
    pomc_targets.csv \
    perturbation_prediction.csv \
    tf_connectivity.csv \
    scenic_adjacencies.csv \
    scenic_regulons.csv \
    scenic_aucell.csv \
    celloracle_pomc_ko_prediction.csv \
    2>/dev/null || echo 'Some files may be missing - check above'

print('ZIP file created: scenic_results.zip')
from google.colab import files
# files.download('scenic_results.zip')